In [ ]:
! pip install datasets transformers

In [2]:
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from pathlib import Path
import json
import os
import torch

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
# Configuration
DATASET_PATH = Path('/content/drive/MyDrive/hf_datasets/italian')
LABEL_MAPPING_PATH = Path('/content/drive/MyDrive/hf_datasets/label_mapping.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/trained_models/itabert')

In [5]:
# Italian model choice
MODEL_NAME = "osiria/bert-italian-cased-ner"

In [6]:
# Training hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 2
MAX_LENGTH = 512

In [7]:
# Load label mapping
print("Loading label mapping...")
with open(LABEL_MAPPING_PATH, 'r') as f:
    label_info = json.load(f)

label2id = {k: int(v) for k, v in label_info["label2id"].items()}
id2label = {int(k): v for k, v in label_info["id2label"].items()}
label_list = label_info["label_list"]

print(f"Labels: {label_list}")
print(f"Number of labels: {len(label_list)}")

Loading label mapping...
Labels: ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-DATE', 'I-DATE']
Number of labels: 9


In [8]:
# Load dataset
print(f"\nLoading dataset from {DATASET_PATH}...")

if not DATASET_PATH.exists():
    print(f"Error: Dataset directory not found at {DATASET_PATH}.")
    print("Please ensure your Google Drive is mounted and the dataset exists at this path.")
else:
    dataset = load_from_disk(DATASET_PATH)

    print(f"Train examples: {len(dataset['train'])}")
    print(f"Val examples: {len(dataset['test'])}")


Loading dataset from /content/drive/MyDrive/hf_datasets/italian...
Train examples: 400
Val examples: 100


In [9]:
# Load tokenizer
print(f"\nLoading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenization function
def tokenize_and_align_labels(examples):
    """
    Tokenize inputs and align NER labels with subword tokens
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens get label -100
            if word_idx is None:
                label_ids.append(-100)
            # First token of a word gets the label
            elif word_idx != previous_word_idx:
                if word_idx < len(label):
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            # Subsequent tokens of the same word get -100
            else:
                label_ids.append(-100)

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize dataset
print("\nTokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(f"Tokenized train examples: {len(tokenized_dataset['train'])}")
print(f"Tokenized val examples: {len(tokenized_dataset['test'])}")

# Load model
print(f"\nLoading model: {MODEL_NAME}...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


Loading tokenizer: osiria/bert-italian-cased-ner...


config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/205k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `5`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: osiria/bert-italian-cased-ner
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5]) vs model:torch.Size([9])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5, 768]) vs model:torch.Size([9, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [10]:
# Metrics

def compute_metrics(p):
    """
    Compute metrics using our own calculations
    """
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens with label -100)
    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        for p, l in zip(prediction, label):
            if l != -100:  # Skip special tokens
                true_predictions.append(id2label[p])
                true_labels.append(id2label[l])

    # Calculate metrics manually
    total_gold = 0
    total_pred = 0
    total_match = 0

    for t, p in zip(true_labels, true_predictions):
        if t != "O":
            total_gold += 1
        if p != "O":
            total_pred += 1
        if t != "O" and t == p:
            total_match += 1

    precision = total_match / total_pred if total_pred > 0 else 0.0
    recall = total_match / total_gold if total_gold > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # Also calculate token-level accuracy
    correct_tokens = sum(1 for t, p in zip(true_labels, true_predictions) if t == p)
    total_tokens = len(true_labels)
    accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
    }

In [13]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    logging_steps=10,
    report_to="none",
    optim="adamw_torch",
    save_total_limit=2,
    dataloader_num_workers=0,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH.name}")
print(f"Output: {OUTPUT_DIR}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print("=" * 60 + "\n")

trainer.train()


STARTING TRAINING
Model: osiria/bert-italian-cased-ner
Dataset: italian
Output: /content/drive/MyDrive/trained_models/itabert
Epochs: 10
Batch size: 8
Learning rate: 2e-05



Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.562829,0.233738,0.586579,0.469444,0.521516,0.930701
2,0.406928,0.193686,0.694904,0.622840,0.656901,0.944670
3,0.339230,0.184364,0.705764,0.650000,0.676735,0.946283
4,0.311221,0.182133,0.722680,0.649074,0.683902,0.948122
5,0.281850,0.179679,0.715007,0.655864,0.684160,0.948155
6,0.255765,0.180078,0.733795,0.653395,0.691265,0.949477
7,0.256250,0.182195,0.723826,0.665741,0.693569,0.949219
8,0.237291,0.182829,0.739840,0.657407,0.696192,0.950026
9,0.266055,0.185850,0.720845,0.663889,0.691195,0.948961
10,0.255541,0.185549,0.726655,0.660494,0.691997,0.949122


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.328714729309082, metrics={'train_runtime': 586.0925, 'train_samples_per_second': 6.825, 'train_steps_per_second': 0.427, 'total_flos': 1045253173248000.0, 'train_loss': 0.328714729309082, 'epoch': 10.0})

In [14]:
# Evaluate
print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

results = trainer.evaluate()

print(f"\nValidation Results:")
print(f"  Precision: {results.get('eval_precision', 0):.4f}")
print(f"  Recall:    {results.get('eval_recall', 0):.4f}")
print(f"  F1 Score:  {results.get('eval_f1', 0):.4f}")
print(f"  Accuracy:  {results.get('eval_accuracy', 0):.4f}")


FINAL EVALUATION


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.255541,0.182829,10,0.739840,0.657407,0.696192,0.950026



Validation Results:
  Precision: 0.7398
  Recall:    0.6574
  F1 Score:  0.6962
  Accuracy:  0.9500


In [15]:
# Save model
print(f"\nSaving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save label mapping with the model
with open(OUTPUT_DIR / "label_mapping.json", 'w') as f:
    json.dump({
        "label_list": label_list,
        "label2id": label2id,
        "id2label": id2label
    }, f, indent=2)

# Save results
with open(OUTPUT_DIR / "eval_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print("\nTraining complete!")
print(f"Model saved to: {OUTPUT_DIR}")


Saving model to /content/drive/MyDrive/trained_models/itabert...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!
Model saved to: /content/drive/MyDrive/trained_models/itabert
